In [1]:
from pathlib import Path
import shutil
import random
import numpy as np
import cv2
from tqdm import tqdm
from scipy import ndimage as ndi

# Notebook is inside:W
# project_root / "Meta Scripts"

meta_scripts_dir = Path.cwd()
project_root = meta_scripts_dir.parent

pngs_dir = project_root / "Data" / "PNGs"
yolo_dir = project_root / "yolo_files" / "yolo_seg"

images_train_dir = yolo_dir / "images" / "train"
images_val_dir = yolo_dir / "images" / "val"
labels_train_dir = yolo_dir / "labels" / "train"
labels_val_dir = yolo_dir / "labels" / "val"

for d in [images_train_dir, images_val_dir, labels_train_dir, labels_val_dir]:
  d.mkdir(parents=True, exist_ok=True)

val_fraction = 0.2
random_seed = 42

class_id = 0
class_name = "bubble"

min_mask_area_px = 5
min_contour_points = 3

# Increase this if YOLO label files are huge.
# Good values to test: 0.002, 0.005, 0.01
polygon_simplification_factor = 0.005

random.seed(random_seed)

In [2]:
def find_cellpose_mask_for_image(image_path):
  """
  Looks for Cellpose output beside the image.

  Expected Cellpose GUI output:
    image.png
    image_seg.npy
  """
  candidates = [
    image_path.with_name(image_path.stem + "_seg.npy"),
    image_path.with_suffix(".npy"),
  ]

  for candidate in candidates:
    if candidate.exists():
      return candidate

  return None


def load_cellpose_masks(seg_npy_path):
  """
  Loads Cellpose *_seg.npy file and returns the integer mask array.

  Expected:
    0 = background
    1, 2, 3, ... = segmented objects
  """
  data = np.load(seg_npy_path, allow_pickle=True).item()

  if "masks" not in data:
    raise KeyError(f"No 'masks' key found in {seg_npy_path}")

  return data["masks"]


def mask_to_yolo_segments_fast(mask_array, image_width, image_height):
  """
  Converts Cellpose integer masks into YOLO-seg polygon annotations.

  YOLO-seg line format:
    class_id x1 y1 x2 y2 x3 y3 ...

  Coordinates are normalized from 0 to 1.
  """
  yolo_lines = []

  mask_array = mask_array.astype(np.int32, copy=False)

  object_slices = ndi.find_objects(mask_array)

  for object_id, object_slice in enumerate(object_slices, start=1):
    if object_slice is None:
      continue

    y_slice, x_slice = object_slice

    y0, y1 = y_slice.start, y_slice.stop
    x0, x1 = x_slice.start, x_slice.stop

    cropped_mask = mask_array[y0:y1, x0:x1]
    binary_mask = (cropped_mask == object_id).astype(np.uint8)

    area = int(binary_mask.sum())
    if area < min_mask_area_px:
      continue

    contours, _ = cv2.findContours(
      binary_mask,
      cv2.RETR_EXTERNAL,
      cv2.CHAIN_APPROX_SIMPLE
    )

    if len(contours) == 0:
      continue

    contour = max(contours, key=cv2.contourArea)

    if len(contour) < min_contour_points:
      continue

    arc_length = cv2.arcLength(contour, True)
    epsilon = polygon_simplification_factor * arc_length
    contour = cv2.approxPolyDP(contour, epsilon, True)

    contour = contour.squeeze()

    if contour.ndim != 2:
      continue

    if contour.shape[0] < min_contour_points:
      continue

    normalized_points = []

    for x, y in contour:
      x_global = x + x0
      y_global = y + y0

      x_norm = float(x_global) / image_width
      y_norm = float(y_global) / image_height

      x_norm = min(max(x_norm, 0.0), 1.0)
      y_norm = min(max(y_norm, 0.0), 1.0)

      normalized_points.extend([x_norm, y_norm])

    if len(normalized_points) >= 6:
      line = str(class_id) + " " + " ".join(f"{p:.6f}" for p in normalized_points)
      yolo_lines.append(line)

  return yolo_lines

In [3]:
image_paths = sorted(
  list(pngs_dir.rglob("*.png")) +
  list(pngs_dir.rglob("*.jpg")) +
  list(pngs_dir.rglob("*.jpeg"))
)

paired_data = []

for image_path in image_paths:
  seg_path = find_cellpose_mask_for_image(image_path)

  if seg_path is not None:
    paired_data.append((image_path, seg_path))

print(f"Found {len(image_paths)} total images")
print(f"Found {len(paired_data)} images with matching Cellpose masks")

for image_path, seg_path in paired_data:
  print("Image:", image_path.relative_to(pngs_dir))
  print("Mask: ", seg_path.relative_to(pngs_dir))
  print()

Found 52 total images
Found 1 images with matching Cellpose masks
Image: 1.2mpm-6LPM-4fps-200us-dB20\1.2mpm-6LPM-200us-20dB-04242026141859-0.png
Mask:  1.2mpm-6LPM-4fps-200us-dB20\1.2mpm-6LPM-200us-20dB-04242026141859-0_seg.npy



In [4]:
random.shuffle(paired_data)

num_val = int(len(paired_data) * val_fraction)

# For very small tests, avoid putting your only image into val.
if len(paired_data) == 1:
  num_val = 0

val_pairs = paired_data[:num_val]
train_pairs = paired_data[num_val:]

print(f"Training images: {len(train_pairs)}")
print(f"Validation images: {len(val_pairs)}")

Training images: 1
Validation images: 0


In [5]:
def process_split(pairs, split_name):
  if split_name == "train":
    output_images_root = images_train_dir
    output_labels_root = labels_train_dir
  elif split_name == "val":
    output_images_root = images_val_dir
    output_labels_root = labels_val_dir
  else:
    raise ValueError("split_name must be 'train' or 'val'")

  for image_path, seg_path in tqdm(pairs, desc=f"Processing {split_name}"):
    relative_image_path = image_path.relative_to(pngs_dir)

    output_image_path = output_images_root / relative_image_path
    output_label_path = output_labels_root / relative_image_path.with_suffix(".txt")

    output_image_path.parent.mkdir(parents=True, exist_ok=True)
    output_label_path.parent.mkdir(parents=True, exist_ok=True)

    shutil.copy2(image_path, output_image_path)

    img = cv2.imread(str(image_path), cv2.IMREAD_UNCHANGED)

    if img is None:
      print(f"Could not read image: {image_path}")
      continue

    image_height, image_width = img.shape[:2]

    masks = load_cellpose_masks(seg_path)

    print()
    print(f"Processing: {image_path.name}")
    print(f"Mask file: {seg_path.name}")
    print(f"Image size: {image_width} x {image_height}")
    print(f"Mask shape: {masks.shape}")
    print(f"Mask dtype: {masks.dtype}")
    print(f"Number of objects: {int(masks.max())}")

    yolo_lines = mask_to_yolo_segments_fast(
      masks,
      image_width,
      image_height
    )

    with open(output_label_path, "w") as f:
      f.write("\n".join(yolo_lines))

    print(f"Wrote label file: {output_label_path}")
    print(f"YOLO polygons written: {len(yolo_lines)}")


process_split(train_pairs, "train")
process_split(val_pairs, "val")

print("Done creating YOLO-seg dataset.")

Processing train:   0%|          | 0/1 [00:00<?, ?it/s]


Processing: 1.2mpm-6LPM-200us-20dB-04242026141859-0.png
Mask file: 1.2mpm-6LPM-200us-20dB-04242026141859-0_seg.npy
Image size: 5320 x 4600
Mask shape: (4600, 5320)
Mask dtype: uint16
Number of objects: 16289


Processing train: 100%|██████████| 1/1 [00:01<00:00,  1.95s/it]


Wrote label file: d:\2026\aspire\yolo_files\yolo_seg\labels\train\1.2mpm-6LPM-4fps-200us-dB20\1.2mpm-6LPM-200us-20dB-04242026141859-0.txt
YOLO polygons written: 16289


Processing val: 0it [00:00, ?it/s]

Done creating YOLO-seg dataset.


In [6]:
yaml_text = f"""path: {yolo_dir.as_posix()}
train: images/train
val: images/val

names:
  0: {class_name}
"""

yaml_path = yolo_dir / "data.yaml"

with open(yaml_path, "w") as f:
  f.write(yaml_text)

print(f"Wrote: {yaml_path}")
print()
print(yaml_text)

Wrote: d:\2026\aspire\yolo_files\yolo_seg\data.yaml

path: d:/2026/aspire/yolo_files/yolo_seg
train: images/train
val: images/val

names:
  0: bubble

